# Õppetund 11 - Agent agentile (A2A) protokoll


## Seadistamine


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv

In [ ]:
import os
import dotenv
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Mis on A2A protokoll?

**Agentidevaheline (A2A) protokoll** on avatud standard, mis võimaldab tehisintellekti agentidel suhelda,
üksteist avastada ja koostööd teha — isegi kui nad on ehitatud erinevatele raamistikudele või majutatud
erinevate teenuste poolt.

Põhimõisted:

- **Avastamine** – Agendid avaldavad *agendikaardi*, mis kirjeldab nende võimeid, muutes
  teistele agentidele (või orkestreerijatele) lihtsaks leida õige spetsialist ülesande täitmiseks.
- **Sõnumite vahetamine** – Agendid vahetavad struktureeritud sõnumeid ühise protokolli kaudu, nii et
  ühe agendi päringut saab mõista ja täita teine olenemata selle sisemisest
  rakendusest.
- **Ülesande elutsükkel** – A2A määratleb seisundid nagu *esitatud*, *töös*, *valmis* ja
  *ebaõnnestunud*, andes orkestreerijale täieliku ülevaate, kuidas delegeeritud ülesanne edeneb.

Selles õppetükis simuleerime A2A-laadset koostööd, ühendades kolm spetsialiseerunud reisibüroo agenti
töövoogu, kus iga agent lisab oma teadmised ja annab tulemused järgmisele.


## Spetsialiseerunud reisikorraldajate loomine


In [ ]:
currency_agent = client.as_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = client.as_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = client.as_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## Mitme agendi koostöö töövoo kaudu

Me ühendame kolm agenti järjestikulisse töövoogu, mis peegeldab A2A sõnumiedastust:

1. **CurrencyExchangeAgent** võtab kasutaja päringu vastu ja annab valuuta juhiseid.
2. **ActivityPlannerAgent** võtab vastu rikastatud konteksti ja lisab tegevuste soovitused.
3. **TravelManagerAgent** sünteesib mõlemad sisendid lõplikuks reisijuhiseks.


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## A2A mõistmine tootmiskeskkonnas

Tootmiskeskkonnas avab A2A protokoll võimsad ristteenuste stsenaariumid:

| Võimekus | Kirjeldus |
|---|---|
| **Raamistikudeülene koostalitlusvõime** | Ühe raamistikuga ehitatud agent saab delegeerida ülesandeid mistahes teise A2A-ühilduva raamistikuga ehitatud agendile, võimaldades tõelist organisatsioonidevahelist koostalitlust. |
| **Teenusepiirid** | Agendid võivad asuda eraldiseisvates mikroteenustes, pilveregioonides või isegi erinevates organisatsioonides, samal ajal sujuvalt koostööd tehes. |
| **Dünaamiline avastamine** | Orkestreerija saab kasutamise ajal pärida Agent Card registrit, et leida antud alamtöö jaoks kõige sobivam spetsialist. |
| **Voogedastus ja push-teavitused** | A2A toetab Server-Sent Events (SSE) reaalajas edenemise uuenduste jaoks ja push-teavitusi pikkade ülesannete korral. |

Ülaltoodud töövoog on selle mustri lihtsustatud, sisesüsteemne versioon. Tõelises
juurutuses ekspordib iga agent HTTP lõpp-punkti, avaldab Agent Cardi ja suhtleb
A2A JSON-RPC protokolli kaudu.


## Summary

In this lesson you learned:

1. **What the A2A protocol is** — an open standard for agent-to-agent discovery, messaging,
   and task management.
2. **How to create specialized agents** — a Currency Exchange agent, an Activity Planner agent,
   and a Travel Manager orchestrator.
3. **How to wire agents into a workflow** — using `WorkflowBuilder` to model sequential
   message passing between agents.
4. **How A2A works in production** — enabling cross-framework, cross-service collaboration
   with dynamic discovery and streaming updates.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Lahtiütlus**:
See dokument on tõlgitud kasutades AI tõlketeenust [Co-op Translator](https://github.com/Azure/co-op-translator). Kuigi me püüdleme täpsuse poole, palun pange tähele, et automatiseeritud tõlgetes võib esineda vigu või ebatäpsusi. Originaaldokument selle emakeeles tuleks pidada autoriteetseks allikaks. Olulise teabe puhul soovitatakse kasutada professionaalset inimtõlget. Me ei vastuta selle tõlkega seotud eksimustest või valesti mõistmistest.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
